# HK-LeadLag - Notebook orchestrateur technique

Ce notebook orchestre les briques techniques du projet Hayashi & Koike (2020) :
1. Briques techniques (wavelet, HY, HK estimator) - voir `notebooks/01_walkthrough.ipynb` pour le détail pédagogique.
2. Monte Carlo de réplication des Tables 2-3 du papier.
3. Application empirique rapide sur données crypto Binance (BTC vs ETH).

**Pour le récit complet** (exploration des données, réplication par étapes, résultats cross-venue, inférence statistique) : ouvrir `notebooks/02_main_narrative.ipynb`, qui est le notebook de démonstration principal.

**Conventions** : tous les modules sont dans le package `hk_leadlag/`, les configs dans `configs/`, les sorties dans `outputs/`.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hk_leadlag.config import ExperimentConfig
from hk_leadlag.analysis.experiment import Experiment
from hk_leadlag.viz.plots import LeadLagPlotter

OUT = Path('outputs')
OUT.mkdir(exist_ok=True)
rng = np.random.default_rng(0)

## 1. Replication Monte Carlo (Tables 2-3 of HK20)

In [ ]:
config = ExperimentConfig.from_yaml('configs/simulation_default.yaml')
# Trim n_paths for first run
config.simulation.n_paths = 100
config.simulation.n_steps = 8192
exp = Experiment(config)
exp.dump_config()
summary = exp.run_monte_carlo()
summary

## 2. Empirical application - Binance BTCUSDT vs ETHUSDT

In [ ]:
from datetime import date
from hk_leadlag.data import BinanceTradesLoader
from hk_leadlag.estimators import WaveletLeadLagEstimator
from hk_leadlag.wavelet import DaubechiesFilter

loader = BinanceTradesLoader(
    symbol1='BTCUSDT', symbol2='ETHUSDT',
    start_date=date(2026, 4, 13), end_date=date(2026, 4, 19),
    market='spot',
)
series = loader.load()
print(f'Loaded BTC ticks: {series.n1:,} | ETH ticks: {series.n2:,}')
print(f'Horizon: {series.T / 3600:.1f} h')

In [ ]:
est = WaveletLeadLagEstimator(
    delta_N=0.01,           # 10 ms
    j_max=8,
    grid_half_width=200,
    daub=DaubechiesFilter('db10'),
)
result = est.fit(series)
for j, t_hat in zip(result.levels, result.theta_hat):
    print(f'j={int(j)}  theta_hat={t_hat*1000:+.1f} ms')

In [ ]:
LeadLagPlotter(figsize=(10, 5)).plot_heatmap_2d(result, save=OUT / 'heatmap_btc_eth.png');
LeadLagPlotter().plot_contrast(result, save=OUT / 'contrast_btc_eth.png');

## 3. 

- **Récit complet de l'analyse** (exploration des données → réplication par étapes → résultats cross-venue → inférence) : `notebooks/02_main_narrative.ipynb`.
- **Runs empiriques reproductibles** : `scripts/run_empirical.py`, `run_cross_exchange.py`, `run_cross_exchange_kraken.py`, `run_bootstrap.py`.
- **Index des résultats par expérience** : `OUTPUTS_INDEX.md`.
- **Pistes d'extension** : `extensions.md`.